# SAE-V Feature Modality Analysis

This notebook provides two types of visualization for SAE-V features:

1. **3D Scatter Plot** — Visualize all 65536 features in the (Modality Ratio, Alignment Score, Frequency) space
2. **Single Feature Deep Dive** — For a chosen feature, show its top activating text tokens and image patches

### Prerequisites
Run `scripts/compute_feature_metrics.py` first to generate `output/feature_metrics/feature_metrics.npz`.

In [ ]:
import os, sys, gc

# Allow MPS to use full unified memory (Apple Silicon)
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display, HTML
from PIL import Image

# Project root
PROJECT_ROOT = os.path.dirname(os.getcwd())
if os.path.basename(os.getcwd()) != 'notebooks':
    PROJECT_ROOT = os.getcwd()  # running from project root
sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

---
## Part 1: 3D Feature Modality Scatter Plot

In [ ]:
# Load pre-computed metrics
metrics_path = os.path.join(PROJECT_ROOT, "output/feature_metrics/feature_metrics.npz")
data = np.load(metrics_path)

modality_ratio = data["modality_ratio"]
alignment      = data["alignment"]
frequency      = data["frequency"]
mean_text      = data["mean_text"]
mean_image     = data["mean_image"]
n_samples      = int(data["n_samples"])

d_sae = len(modality_ratio)
alive  = frequency > 0
n_alive = alive.sum()

print(f"Total features: {d_sae}")
print(f"Alive features: {n_alive}  ({n_alive/d_sae*100:.1f}%)")
print(f"Samples used:   {n_samples}")
print(f"\nModality Ratio  — range [{modality_ratio[alive].min():.3f}, {modality_ratio[alive].max():.3f}]")
print(f"Alignment Score — range [{alignment[alive].min():.3f}, {alignment[alive].max():.3f}]")
print(f"Frequency       — range [{frequency[alive].min():.3f}, {frequency[alive].max():.3f}]")

In [ ]:
# Classify features
def classify_features(mr, al, freq, ratio_thresh=0.8, freq_thresh=0.02):
    """Classify each alive feature into text / visual / cross-modal / rare."""
    cats = np.full(len(mr), 'dead', dtype=object)
    alive_mask = freq > 0
    for i in np.where(alive_mask)[0]:
        if freq[i] < freq_thresh:
            cats[i] = 'rare'
        elif mr[i] > ratio_thresh:
            cats[i] = 'visual'
        elif mr[i] < -ratio_thresh:
            cats[i] = 'text'
        else:
            cats[i] = 'cross-modal'
    return cats

categories = classify_features(modality_ratio, alignment, frequency)

for cat in ['text', 'visual', 'cross-modal', 'rare', 'dead']:
    n = (categories == cat).sum()
    print(f"  {cat:>12s}: {n:6d}  ({n/d_sae*100:5.1f}%)")

In [ ]:
# Interactive 3D scatter with Plotly
import plotly.graph_objects as go

mr_a = modality_ratio[alive]
al_a = alignment[alive]
fr_a = frequency[alive]
idx_a = np.where(alive)[0]
cat_a = categories[alive]

fig = go.Figure()

cat_info = [
    ('text',        '#3498db', 'Text-dominant'),
    ('visual',      '#e74c3c', 'Image-dominant'),
    ('cross-modal', '#2ecc71', 'Cross-modal'),
    ('rare',        '#95a5a6', 'Rare'),
]

for cat, col, label in cat_info:
    mask = cat_a == cat
    if mask.sum() == 0:
        continue
    hover = [
        f"Feature {idx_a[j]}<br>"
        f"Ratio: {mr_a[j]:.3f}<br>"
        f"Alignment: {al_a[j]:.3f}<br>"
        f"Frequency: {fr_a[j]:.3f}"
        for j in np.where(mask)[0]
    ]
    fig.add_trace(go.Scatter3d(
        x=mr_a[mask], y=al_a[mask], z=fr_a[mask],
        mode='markers',
        marker=dict(size=2.5, color=col, opacity=0.6),
        name=f"{label} ({mask.sum()})",
        text=hover, hoverinfo='text',
    ))

fig.update_layout(
    title='SAE-V Feature Modality Analysis',
    scene=dict(
        xaxis_title='Modality Ratio (\u2190 text | image \u2192)',
        yaxis_title='Alignment Score',
        zaxis_title='Activation Frequency',
    ),
    width=950, height=700,
    legend=dict(x=0.01, y=0.99),
)
fig.show()

In [ ]:
# 2D marginal projections
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

color_map = {'text': '#3498db', 'visual': '#e74c3c',
             'cross-modal': '#2ecc71', 'rare': '#95a5a6'}

for cat, col in color_map.items():
    mask = cat_a == cat
    if mask.sum() == 0:
        continue
    axes[0].scatter(mr_a[mask], al_a[mask], c=col, s=3, alpha=0.4, label=cat)
    axes[1].scatter(mr_a[mask], fr_a[mask], c=col, s=3, alpha=0.4, label=cat)
    axes[2].scatter(al_a[mask], fr_a[mask], c=col, s=3, alpha=0.4, label=cat)

axes[0].set_xlabel('Modality Ratio'); axes[0].set_ylabel('Alignment Score')
axes[0].set_title('Ratio vs Alignment')
axes[1].set_xlabel('Modality Ratio'); axes[1].set_ylabel('Frequency')
axes[1].set_title('Ratio vs Frequency')
axes[2].set_xlabel('Alignment Score'); axes[2].set_ylabel('Frequency')
axes[2].set_title('Alignment vs Frequency')

for ax in axes:
    ax.legend(fontsize=8, markerscale=3)
    ax.grid(True, alpha=0.3)

plt.suptitle('2D Marginal Projections', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram of modality ratios for alive features
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(mr_a, bins=100, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Modality Ratio'); axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Modality Ratio')
axes[0].axvline(0, color='red', ls='--', alpha=0.5)

axes[1].hist(al_a[al_a != 0], bins=100, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Alignment Score'); axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Alignment Score')

axes[2].hist(fr_a, bins=100, color='mediumseagreen', edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Frequency'); axes[2].set_ylabel('Count')
axes[2].set_title('Distribution of Activation Frequency')

plt.tight_layout()
plt.show()

---
## Part 2: Single Feature Deep Dive

Select a feature of interest and visualize:
- Which text tokens activate it (with colored intensity)
- Which image patches activate it (heatmap overlay on the original image)
- Activation value statistics

In [ ]:
# Pick interesting features from each category
def top_features_by_category(categories, frequency, modality_ratio, alignment, k=5):
    """Find top-k most frequent features per category."""
    results = {}
    for cat in ['text', 'visual', 'cross-modal']:
        mask = categories == cat
        if mask.sum() == 0:
            results[cat] = []
            continue
        idxs = np.where(mask)[0]
        top_k = idxs[np.argsort(-frequency[idxs])[:k]]
        results[cat] = top_k.tolist()
    return results

top_feats = top_features_by_category(categories, frequency, modality_ratio, alignment)

for cat, feats in top_feats.items():
    print(f"\n{cat.upper()} features (top by frequency):")
    for f in feats:
        print(f"  Feature {f:5d}  ratio={modality_ratio[f]:+.3f}  "
              f"align={alignment[f]:.3f}  freq={frequency[f]:.3f}")

In [ ]:
# ============================================================
# Load models (run this cell once — takes ~1 min on MPS)
# ============================================================

from sae_lens import SAE
from transformers import LlavaNextForConditionalGeneration, LlavaNextProcessor
from transformer_lens.HookedLlava import HookedLlava

def get_device():
    if torch.backends.mps.is_available(): return 'mps'
    elif torch.cuda.is_available(): return 'cuda:0'
    return 'cpu'

DEVICE = get_device()
DTYPE = torch.float16
MODEL_PATH = os.path.join(PROJECT_ROOT, 'model/llava-v1.6-mistral-7b-hf')
SAE_PATH   = os.path.join(PROJECT_ROOT, 'model/SAE-V/SAEV_LLaVA_NeXT-7b_OBELICS')

print(f'Device: {DEVICE},  dtype: {DTYPE}')
print(f'Loading models ...')

processor = LlavaNextProcessor.from_pretrained(MODEL_PATH)

vision_model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_PATH, torch_dtype=DTYPE, low_cpu_mem_usage=True)

hook_lm = HookedLlava.from_pretrained_no_processing(
    'llava-hf/llava-v1.6-mistral-7b-hf',
    hf_model=vision_model.language_model,
    device='cpu', fold_ln=False, center_writing_weights=False,
    center_unembed=False, tokenizer=None, dtype=DTYPE,
    vision_tower=vision_model.vision_tower,
    multi_modal_projector=vision_model.multi_modal_projector,
    n_devices=1)
del vision_model; gc.collect()

hook_lm = hook_lm.to(DEVICE); gc.collect()
if DEVICE == 'mps': torch.mps.empty_cache()

sae = SAE.load_from_pretrained(path=SAE_PATH, device=DEVICE)
sae.cfg.device = DEVICE

HOOK_NAME = sae.cfg.hook_name
STOP_LAYER = int(HOOK_NAME.split('.')[1]) + 1
print(f'Models loaded.  hook={HOOK_NAME}  stop_layer={STOP_LAYER}')

In [ ]:
# ============================================================
# Helper functions for single-feature visualization
# ============================================================

IMAGE_TOKEN_ID = 32000  # <image> token for LLaVA-NeXT Mistral

def prepare_input_for_vis(proc, image_pil, question, device):
    """Prepare model input (RLAIF-V style prompt)."""
    prompt = f" USER: \n<image> {question}\nASSISTANT: "
    img = image_pil.resize((336, 336)).convert('RGB')
    text_inp = proc.tokenizer(prompt, return_tensors='pt')
    img_inp  = proc.image_processor(images=img, return_tensors='pt')
    inputs = {
        'input_ids':      text_inp['input_ids'],
        'attention_mask':  text_inp['attention_mask'],
        'pixel_values':   img_inp['pixel_values'],
        'image_sizes':    img_inp['image_sizes'],
    }
    return {k: v.to(device) for k, v in inputs.items()}, img


def run_sae_on_sample(inputs, hook_lm, sae, hook_name, stop_layer, device):
    """Run model + SAE, return feature_acts, hidden_states, image_indice."""
    with torch.no_grad():
        out, cache = hook_lm.run_with_cache(
            input=inputs, model_inputs=inputs, vision=True,
            prepend_bos=True,
            names_filter=lambda name: name == hook_name,
            stop_at_layer=stop_layer)
        image_indice = out[1]  # (1, 1176)
        hidden_states = cache[hook_name].to(device)
        feature_acts = sae.encode(hidden_states)
        del cache, out
    return feature_acts, hidden_states, image_indice


def get_text_tokens(proc, question):
    """Get the list of text token strings (excluding <image>)."""
    prompt = f" USER: \n<image> {question}\nASSISTANT: "
    token_ids = proc.tokenizer.encode(prompt, add_special_tokens=True)
    tokens = []
    for tid in token_ids:
        if tid == IMAGE_TOKEN_ID:
            continue
        tokens.append((tid, proc.tokenizer.decode(tid)))
    return tokens


def visualize_single_feature(feature_idx, feature_acts, image_indice,
                             image_pil, question, processor,
                             modality_ratio=None, alignment=None, frequency=None):
    """
    Visualize a single SAE feature's activations across text and image tokens.
    
    Shows:
      - Text tokens colored by activation intensity
      - Image patch activation heatmap overlay
      - Activation statistics
    """
    fa = feature_acts[0, :, feature_idx].float().cpu()  # (seq_len,)
    seq_len = fa.shape[0]

    # Separate text / image positions
    img_pos = set(image_indice[0].cpu().tolist())
    text_pos = sorted(set(range(seq_len)) - img_pos)
    img_pos_sorted = sorted(img_pos)

    text_activations = fa[text_pos].numpy()
    image_activations = fa[img_pos_sorted].numpy()

    # Get text token strings
    text_tokens = get_text_tokens(processor, question)

    # === Figure ===
    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)

    # -- Header info --
    info = f"Feature {feature_idx}"
    if modality_ratio is not None:
        info += f"  |  Ratio={modality_ratio[feature_idx]:+.3f}"
    if alignment is not None:
        info += f"  |  Align={alignment[feature_idx]:.3f}"
    if frequency is not None:
        info += f"  |  Freq={frequency[feature_idx]:.3f}"
    fig.suptitle(info, fontsize=14, fontweight='bold')

    # -- 1. Text token activation bar chart --
    ax1 = fig.add_subplot(gs[0, :])
    n_text = min(len(text_tokens), len(text_activations))
    colors = plt.cm.Reds(text_activations[:n_text] / (text_activations.max() + 1e-8))
    ax1.bar(range(n_text), text_activations[:n_text], color=colors, edgecolor='gray', lw=0.5)
    ax1.set_xticks(range(n_text))
    ax1.set_xticklabels([t[1] for t in text_tokens[:n_text]],
                        rotation=45, ha='right', fontsize=8)
    ax1.set_ylabel('Activation')
    ax1.set_title('Text Token Activations')
    ax1.grid(axis='y', alpha=0.3)

    # -- 2. Original image --
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.imshow(image_pil)
    ax2.set_title('Original Image')
    ax2.axis('off')

    # -- 3. Image patch heatmap --
    ax3 = fig.add_subplot(gs[1, 1])
    # Try to reshape image activations to a 2D grid
    n_img = len(image_activations)
    # LLaVA-NeXT: 1176 = 576*2 + 24 newlines, use first 576 as 24x24
    if n_img >= 576:
        patch_acts = image_activations[:576].reshape(24, 24)
    else:
        side = int(np.sqrt(n_img))
        patch_acts = image_activations[:side*side].reshape(side, side)
    
    im = ax3.imshow(patch_acts, cmap='hot', interpolation='nearest')
    ax3.set_title('Image Patch Activations')
    ax3.axis('off')
    plt.colorbar(im, ax=ax3, fraction=0.046)

    # -- 4. Overlay --
    ax4 = fig.add_subplot(gs[1, 2])
    # Upsample patch heatmap to image size
    patch_up = np.repeat(np.repeat(patch_acts, 14, axis=0), 14, axis=1)
    img_arr = np.array(image_pil.resize((336, 336)))
    ax4.imshow(img_arr, alpha=0.6)
    ax4.imshow(patch_up, cmap='hot', alpha=0.4)
    ax4.set_title('Activation Overlay')
    ax4.axis('off')

    plt.tight_layout()
    plt.show()

    # -- 5. HTML text highlight --
    max_act = text_activations.max() + 1e-8
    html_parts = ['<div style="font-family:monospace; font-size:14px; line-height:2; '
                  'background:#1a1a1a; padding:12px; border-radius:8px;">']
    html_parts.append('<b style="color:#ccc;">Text tokens (colored by activation):</b><br>')
    for i, (tid, tok_str) in enumerate(text_tokens[:n_text]):
        act_val = text_activations[i] if i < len(text_activations) else 0
        intensity = act_val / max_act
        r = int(255 * intensity)
        g = int(80 * (1 - intensity))
        b = int(80 * (1 - intensity))
        bg_alpha = 0.3 + 0.7 * intensity
        html_parts.append(
            f'<span style="background:rgba({r},{g},{b},{bg_alpha:.2f}); '
            f'color:white; padding:2px 4px; margin:1px; border-radius:3px; '
            f'display:inline-block;" '
            f'title="act={act_val:.2f}">{tok_str}</span> ')
    html_parts.append('</div>')
    display(HTML(''.join(html_parts)))

    # Stats
    print(f"\n--- Feature {feature_idx} Stats ---")
    print(f"  Text  — mean: {text_activations.mean():.3f}, max: {text_activations.max():.3f}, "
          f"active: {(text_activations > 0).sum()}/{len(text_activations)}")
    print(f"  Image — mean: {image_activations.mean():.3f}, max: {image_activations.max():.3f}, "
          f"active: {(image_activations > 0).sum()}/{len(image_activations)}")

In [ ]:
# ============================================================
# Run on a sample and visualize features
# ============================================================

# --- Load a sample from RLAIF-V ---
from datasets import load_dataset

ds = load_dataset('openbmb/RLAIF-V-Dataset', split='train[:5]')
SAMPLE_IDX = 0  # <-- change this to try different samples
sample = ds[SAMPLE_IDX]

question = sample['question']
image_pil = sample['image']
print(f'Question: {question}')
display(image_pil.resize((300, 300)))

In [ ]:
# --- Run model + SAE ---
inputs, img_resized = prepare_input_for_vis(processor, image_pil, question, DEVICE)

feature_acts, hidden_states, image_indice = run_sae_on_sample(
    inputs, hook_lm, sae, HOOK_NAME, STOP_LAYER, DEVICE)

print(f'Feature acts shape: {feature_acts.shape}')
print(f'Image indices shape: {image_indice.shape}')
print(f'Sequence length: {feature_acts.shape[1]}')
print(f'Text tokens:  {feature_acts.shape[1] - image_indice.shape[1]}')
print(f'Image tokens: {image_indice.shape[1]}')

In [ ]:
# ============================================================
# Visualize specific features
# ============================================================

# Option 1: Use a feature from the top_feats analysis above
# Option 2: Pick the top-activated feature on this specific sample

# Find top features on this sample
total_act = feature_acts[0].float().sum(dim=0).cpu()  # sum across tokens
top_vals, top_idxs = torch.topk(total_act, 10)

print("Top 10 most activated features on this sample:")
for v, i in zip(top_vals, top_idxs):
    cat_str = categories[i.item()] if i.item() < len(categories) else '?'
    print(f"  Feature {i.item():5d}  total_act={v.item():.1f}  category={cat_str}")

In [ ]:
# ============================================================
# Visualize feature #1 — the top activated feature
# ============================================================
FEATURE_IDX = top_idxs[0].item()  # <-- change this to explore other features

visualize_single_feature(
    FEATURE_IDX, feature_acts, image_indice,
    img_resized, question, processor,
    modality_ratio=modality_ratio,
    alignment=alignment,
    frequency=frequency,
)

In [ ]:
# ============================================================
# Compare features from different categories
# ============================================================

# Find one text-dominant, one image-dominant, one cross-modal feature
# that are active on this sample
fa_sum = feature_acts[0].float().sum(dim=0).cpu().numpy()

for cat_name in ['text', 'visual', 'cross-modal']:
    cat_mask = categories == cat_name
    active_in_cat = cat_mask & (fa_sum > 0)
    if active_in_cat.sum() == 0:
        print(f"\nNo active {cat_name} feature on this sample.")
        continue
    # Pick the one with highest activation in this sample
    candidates = np.where(active_in_cat)[0]
    best = candidates[np.argmax(fa_sum[candidates])]
    print(f"\n{'='*60}")
    print(f"Category: {cat_name.upper()}  —  Feature {best}")
    print(f"{'='*60}")
    visualize_single_feature(
        best, feature_acts, image_indice,
        img_resized, question, processor,
        modality_ratio=modality_ratio,
        alignment=alignment,
        frequency=frequency,
    )

In [ ]:
# ============================================================
# Multi-sample feature analysis
# Run on multiple samples to see how a feature behaves
# ============================================================

FEATURE_TO_ANALYZE = top_idxs[0].item()  # <-- change this
N_VIS_SAMPLES = 3  # number of samples to compare

fig, axes = plt.subplots(N_VIS_SAMPLES, 3, figsize=(15, 5 * N_VIS_SAMPLES))
if N_VIS_SAMPLES == 1:
    axes = axes[np.newaxis, :]

for s_idx in range(min(N_VIS_SAMPLES, len(ds))):
    sample_s = ds[s_idx]
    inp_s, img_s = prepare_input_for_vis(processor, sample_s['image'],
                                         sample_s['question'], DEVICE)
    fa_s, _, ii_s = run_sae_on_sample(inp_s, hook_lm, sae, HOOK_NAME, STOP_LAYER, DEVICE)

    act_s = fa_s[0, :, FEATURE_TO_ANALYZE].float().cpu().numpy()
    img_pos_s = set(ii_s[0].cpu().tolist())
    img_pos_sorted_s = sorted(img_pos_s)
    img_acts_s = act_s[img_pos_sorted_s]

    # Original image
    axes[s_idx, 0].imshow(img_s)
    axes[s_idx, 0].set_title(f'Sample {s_idx}', fontsize=10)
    axes[s_idx, 0].axis('off')

    # Patch heatmap
    if len(img_acts_s) >= 576:
        patch = img_acts_s[:576].reshape(24, 24)
    else:
        side = int(np.sqrt(len(img_acts_s)))
        patch = img_acts_s[:side*side].reshape(side, side)
    im = axes[s_idx, 1].imshow(patch, cmap='hot', interpolation='nearest')
    axes[s_idx, 1].set_title(f'Feature {FEATURE_TO_ANALYZE} patches', fontsize=10)
    axes[s_idx, 1].axis('off')
    plt.colorbar(im, ax=axes[s_idx, 1], fraction=0.046)

    # Overlay
    patch_up = np.repeat(np.repeat(patch, 14, axis=0), 14, axis=1)
    axes[s_idx, 2].imshow(np.array(img_s), alpha=0.6)
    axes[s_idx, 2].imshow(patch_up, cmap='hot', alpha=0.4)
    axes[s_idx, 2].set_title('Overlay', fontsize=10)
    axes[s_idx, 2].axis('off')

    # Cleanup
    del fa_s, inp_s, ii_s; gc.collect()
    if DEVICE == 'mps': torch.mps.empty_cache()

plt.suptitle(f'Feature {FEATURE_TO_ANALYZE} across samples', fontsize=14)
plt.tight_layout()
plt.show()